# K-Means

## 4. Hàm chung

In [69]:
def calinski_harabasz_score_manual(X, labels):
    """
    Tính Calinski-Harabasz Score.
    """

    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    unique_clusters = np.unique(labels)
    n_samples = X.shape[0]
    K = len(unique_clusters)

    if K < 2 or K >= n_samples:
        return np.nan

    overall_mean = np.mean(X, axis=0)

    B = 0  # between-cluster dispersion
    W = 0  # within-cluster dispersion

    for cluster in unique_clusters:
        Xk = X[labels == cluster]
        nk = Xk.shape[0]

        centroid = np.mean(Xk, axis=0)

        # Between-cluster
        B += nk * np.sum((centroid - overall_mean) ** 2)

        # Within-cluster
        W += np.sum((Xk - centroid) ** 2)

    if W == 0:
        return np.inf

    CH = (B / (K - 1)) / (W / (n_samples - K))

    return CH

In [70]:
# Đánh giá mẫu dữ liệu đầu vào: 
def evaluate_internal(X, labels, model_name='KMeans'):
    """ 
    Đánh giá internal metrics.
    """
    sil = silhouette_score_manual(X, labels)
    db = davies_bouldin_score_manual(X, labels)
    ch = calinski_harabasz_score_manual(X, labels)

    print(f"\n{'='*50}")
    print(f"  Internal Metrics — {model_name}")
    print(f"{'='*50}")
    print(f"  Silhouette Score    : {sil:+.4f}  ")
    print(f"  Davies-Bouldin Index: {db:.4f}   ")
    print(f"  Calinski-Harabasz   : {ch:.1f} ")
    print(f"{'='*50}\n")

    return {'model': model_name, 'silhouette': sil,
            'davies_bouldin': db, 'calinski_harabasz': ch}

In [71]:
def comb2(n):
    """
    Tính tổ hợp C(n, 2) = n * (n - 1) / 2
    """
    return n * (n - 1) / 2


def contingency_matrix(y_true, y_pred):
    """
    Tạo bảng contingency giữa nhãn thật y_true và nhãn cụm y_pred.

    Hàng: các lớp thật
    Cột: các cụm dự đoán
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    true_classes = np.unique(y_true)
    pred_classes = np.unique(y_pred)
    table = np.zeros((len(true_classes), len(pred_classes)), dtype=int)
    for i, true_label in enumerate(true_classes):
        for j, pred_label in enumerate(pred_classes):
            table[i, j] = np.sum((y_true == true_label) & (y_pred == pred_label))
    return table

In [72]:
def adjusted_rand_score_manual(y_true, y_pred):
    """
    Tính Adjusted Rand Index thủ công.

    ARI đo mức độ giống nhau giữa:
    - nhãn thật / nhãn target đã chia bin
    - nhãn cụm sinh ra từ thuật toán phân cụm

    Giá trị:
    - Gần 1: phân cụm rất giống nhãn thật
    - Gần 0: gần như ngẫu nhiên
    - Âm: tệ hơn ngẫu nhiên
    """

    contingency = contingency_matrix(y_true, y_pred)
    n = np.sum(contingency)
    sum_comb = np.sum([comb2(n_ij) for n_ij in contingency.flatten()])
    row_sums = np.sum(contingency, axis=1)
    col_sums = np.sum(contingency, axis=0)
    sum_comb_rows = np.sum([comb2(n_i) for n_i in row_sums])
    sum_comb_cols = np.sum([comb2(n_j) for n_j in col_sums])
    total_comb = comb2(n)

    if total_comb == 0:
        return 0
    expected_index = (sum_comb_rows * sum_comb_cols) / total_comb
    max_index = 0.5 * (sum_comb_rows + sum_comb_cols)
    denominator = max_index - expected_index
    if denominator == 0:
        return 0
    ari = (sum_comb - expected_index) / denominator

    return ari

In [73]:
def fowlkes_mallows_score_manual(y_true, y_pred):
    """
    Tính Fowlkes-Mallows Index thủ công.

    FMI = TP / sqrt((TP + FP) * (TP + FN))

    Giá trị:
    - Gần 1: phân cụm tốt
    - Gần 0: phân cụm kém
    """

    contingency = contingency_matrix(y_true, y_pred)
    tp = np.sum([comb2(n_ij) for n_ij in contingency.flatten()])
    row_sums = np.sum(contingency, axis=1)
    col_sums = np.sum(contingency, axis=0)
    tp_fn = np.sum([comb2(n_i) for n_i in row_sums])
    tp_fp = np.sum([comb2(n_j) for n_j in col_sums])

    if tp_fp == 0 or tp_fn == 0:
        return 0
    fmi = tp / np.sqrt(tp_fp * tp_fn)
    return fmi

In [74]:
def get_bin_labels(n_bins):
    """
    Tạo nhãn cho các mức target.
    """

    if n_bins == 3:
        return ["Thấp", "Trung bình", "Cao"]
    elif n_bins == 4:
        return ["Thấp", "Trung bình thấp", "Trung bình cao", "Cao"]
    else:
        return [f"Mức {i + 1}" for i in range(n_bins)]

In [75]:
def evaluate_external(df, labels, target_col="charges", n_bins=3, model_name="KMeans"):
    """
    Đánh giá quan hệ giữa kết quả phân cụm và biến đầu ra.

    Dùng được cho:
    - K-Means
    - K-Prototypes
    - các thuật toán phân cụm khác

    - df chứa target_col
    - labels là nhãn cụm
    """

    df = df.copy()
    labels = np.asarray(labels)

    df["_cluster"] = labels

    # Rời rạc hóa target_col thành n_bins mức theo quantile
    bin_labels = get_bin_labels(n_bins)

    df["_target_bin"] = pd.qcut(
        df[target_col],
        q=n_bins,
        labels=bin_labels,
        duplicates="drop"
    )

    # Chuyển sang string để tránh lỗi kiểu category
    y_true = df["_target_bin"].astype(str).values
    y_pred = labels

    # Tính ARI, FMI thủ công
    ari = adjusted_rand_score_manual(y_true, y_pred)
    fmi = fowlkes_mallows_score_manual(y_true, y_pred)

    # Trung bình target_col theo từng cụm
    mean_target = df.groupby("_cluster")[target_col].mean().sort_values()

    print(f"\n{'='*50}")
    print(f"  External Metrics — {model_name} vs '{target_col}'")
    print(f"{'='*50}")
    print(f"  ARI (Adjusted Rand Index)  : {ari:+.4f}  (>0 = tốt hơn ngẫu nhiên)")
    print(f"  FMI (Fowlkes-Mallows Index): {fmi:.4f}   (0→1, cao hơn tốt hơn)")

    print(f"\n  Trung bình '{target_col}' theo cụm:")
    for cid, val in mean_target.items():
        print(f"    Cluster {cid}: {val:.4f}")

    print(f"\n  Crosstab cụm × mức {target_col}:")
    print(pd.crosstab(df["_cluster"], df["_target_bin"], margins=True))

    print(f"{'='*50}\n")

    return {
        "model": model_name,
        "ari": ari,
        "fmi": fmi,
        "mean_target_per_cluster": mean_target.to_dict()
    }

### Display

In [76]:
# =====================================================
# 1. PCA plot
# =====================================================

def plot_clusters_pca(
    X,
    labels,
    km_model=None,
    title="KMeans",
    ax=None
):
    """
    Vẽ phân cụm trên không gian PCA 2D.

    Dùng được cho:
    - KMeans
    - KPrototypes
    - các thuật toán khác có labels

    Lưu ý:
    - Với KPrototypes: truyền km_model=None.
    - Với KMeans sklearn: nếu muốn vẽ centroid thì truyền km_model.
    """

    X_arr = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    k = len(np.unique(labels))

    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(X_arr)
    var = pca.explained_variance_ratio_

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 5))

    for c in np.unique(labels):
        mask = labels == c
        ax.scatter(
            X2[mask, 0],
            X2[mask, 1],
            color=COLORS[int(c) % len(COLORS)],
            alpha=0.45,
            s=12,
            label=f"Cluster {c}"
        )

    # Chỉ vẽ centroid nếu là model có cluster_centers_
    # KPrototypes không có cluster_centers_ dạng sklearn nên bỏ qua
    if km_model is not None and hasattr(km_model, "cluster_centers_"):
        centroids_2d = pca.transform(km_model.cluster_centers_)
        ax.scatter(
            centroids_2d[:, 0],
            centroids_2d[:, 1],
            color="black",
            marker="X",
            s=200,
            zorder=5,
            label="Centroid"
        )

    ax.set_title(
        f"{title} – PCA 2D\n(PC1={var[0]:.1%}, PC2={var[1]:.1%})",
        fontweight="bold",
        fontsize=10
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(fontsize=8, markerscale=1.5)
    ax.grid(True, alpha=0.2)

    if standalone:
        plt.tight_layout()
        plt.show()


# =====================================================
# 2. UMAP plot
# =====================================================

def plot_clusters_umap(
    X,
    labels,
    km_model=None,
    title="KMeans",
    ax=None
):
    """
    Vẽ phân cụm trên không gian UMAP 2D.

    Dùng được cho:
    - KMeans
    - KPrototypes
    - các thuật toán khác có labels

    Lưu ý:
    - Với KPrototypes: truyền km_model=None.
    - UMAP chủ yếu dùng để visualize labels, không phải để đánh giá chính.
    """

    import umap

    X_arr = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    reducer = umap.UMAP(n_components=2, random_state=42)
    X2 = reducer.fit_transform(X_arr)

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 5))

    for c in np.unique(labels):
        mask = labels == c
        ax.scatter(
            X2[mask, 0],
            X2[mask, 1],
            color=COLORS[int(c) % len(COLORS)],
            alpha=0.45,
            s=12,
            label=f"Cluster {c}"
        )

    # Với UMAP, không nên vẽ centroid quá tin tưởng.
    # Nếu muốn đơn giản và ổn định thì bỏ centroid.
    # Để an toàn, không vẽ centroid cho mọi model.

    ax.set_title(f"{title} – UMAP 2D", fontweight="bold", fontsize=10)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(fontsize=8, markerscale=1.5)
    ax.grid(True, alpha=0.2)

    if standalone:
        plt.tight_layout()
        plt.show()


# =====================================================
# 3. Phân phối kích thước cụm
# =====================================================

def plot_cluster_distribution(
    labels,
    title="KMeans",
    ax=None
):
    """
    Pie chart thể hiện tỉ lệ mẫu trong từng cụm.

    Dùng được cho cả KMeans và KPrototypes
    vì chỉ cần labels.
    """

    labels = np.asarray(labels)
    clusters = sorted(np.unique(labels))
    sizes = [np.sum(labels == c) for c in clusters]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 5))

    ax.pie(
        sizes,
        labels=[f"Cluster {c}\n({s:,})" for c, s in zip(clusters, sizes)],
        colors=[COLORS[int(c) % len(COLORS)] for c in clusters],
        autopct="%1.1f%%",
        startangle=140,
        wedgeprops=dict(edgecolor="white", linewidth=1.5)
    )

    ax.set_title(f"Phân phối kích thước cụm — {title}", fontweight="bold")

    if standalone:
        plt.tight_layout()
        plt.show()


# =====================================================
# 4. Heatmap đặc trưng theo cụm
# =====================================================

def plot_feature_heatmap(
    df,
    labels,
    feature_names,
    top_n=20,
    title="KMeans",
    ax=None
):
    """
    Heatmap thể hiện trung bình đặc trưng theo cụm.

    Dùng được cho:
    - KMeans
    - KPrototypes

    Ý nghĩa:
    - Biến numeric đã chuẩn hóa: mean cho biết cụm cao/thấp ở feature đó.
    - Biến 0/1: mean chính là tỉ lệ mẫu có giá trị 1 trong cụm.
    """

    labels = np.asarray(labels)
    clusters = sorted(np.unique(labels))

    df = df.copy()
    df["_cluster"] = labels

    feats = [f for f in feature_names if f in df.columns]

    cluster_means = np.array([
        df[df["_cluster"] == c][feats].mean().values
        for c in clusters
    ])

    feat_std = cluster_means.std(axis=0)

    if len(feat_std) == 0:
        print("Không có feature hợp lệ để vẽ heatmap.")
        return

    top_idx = np.argsort(feat_std)[::-1][:top_n]
    top_feats = [feats[i] for i in top_idx]
    data = cluster_means[:, top_idx]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(max(14, top_n * 0.7), 3 + len(clusters)))

    im = ax.imshow(
        data,
        aspect="auto",
        cmap="RdYlGn",
        vmin=-1.5,
        vmax=1.5
    )

    ax.set_xticks(range(len(top_feats)))
    ax.set_xticklabels(top_feats, rotation=40, ha="right", fontsize=8)

    ax.set_yticks(range(len(clusters)))
    ax.set_yticklabels(
        [f"Cluster {c} (n={np.sum(labels == c):,})" for c in clusters],
        fontsize=10
    )

    ax.set_title(
        f"Heatmap — Top {top_n} đặc trưng phân biệt — {title}",
        fontweight="bold"
    )

    for i in range(len(clusters)):
        for j in range(len(top_feats)):
            ax.text(
                j,
                i,
                f"{data[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=6,
                color="black"
            )

    plt.colorbar(im, ax=ax, label="Giá trị trung bình / chuẩn hóa")

    if standalone:
        plt.tight_layout()
        plt.show()

In [77]:
def full_cluster_report(
    X,
    df,
    labels,
    km_model=None,
    feature_names=None,
    df_for_heatmap=None,
    target_col="charges",
    model_name="KMeans",
    use_umap=False,
    top_n_features=20,
    internal=None,
    n_bins=3
):
    """
    X              : dữ liệu dùng để vẽ PCA/UMAP và tính internal nếu là KMeans
    df             : DataFrame gốc có chứa target_col
    labels         : nhãn cụm
    km_model       : model KMeans, nếu KPrototypes thì để None
    feature_names  : tên feature gốc dùng để vẽ heatmap
    df_for_heatmap : DataFrame feature gốc sau tiền xử lý
    target_col     : biến đầu ra để đánh giá external
    model_name     : tên mô hình
    use_umap       : True nếu muốn vẽ scatter bằng UMAP
    top_n_features : số feature hiển thị trên heatmap
    internal       : nếu là KPrototypes thì truyền internal đã tính từ class KPrototypes vào
    n_bins         : số mức chia target_col
    """

    if df_for_heatmap is None:
        df_for_heatmap = df.copy()

    if feature_names is None:
        feature_names = df_for_heatmap.columns.tolist()

    feature_names = list(feature_names)

    # =====================================================
    # Internal metrics
    # =====================================================
    # Nếu internal chưa truyền vào thì mặc định dùng evaluate_internal của KMeans
    # Với KPrototypes: nên truyền internal = kproto_model.evaluate_internal(...)
    if internal is None:
        internal = evaluate_internal(X, labels, model_name)

    # =====================================================
    # External metrics dùng chung
    # =====================================================
    external = evaluate_external(
        df=df,
        labels=labels,
        target_col=target_col,
        n_bins=n_bins,
        model_name=model_name
    )

    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    fig.suptitle(f"Cluster Report — {model_name}", fontsize=15, fontweight="bold")

    # =====================================================
    # [0,0] Scatter PCA hoặc UMAP
    # =====================================================
    # KPrototypes thì truyền km_model=None để không vẽ centroid.
    if use_umap:
        plot_clusters_umap(
            X,
            labels,
            km_model=km_model,
            title=model_name,
            ax=axes[0, 0]
        )
    else:
        plot_clusters_pca(
            X,
            labels,
            km_model=km_model,
            title=model_name,
            ax=axes[0, 0]
        )

    # =====================================================
    # [0,1] Phân phối kích thước cụm
    # =====================================================
    plot_cluster_distribution(labels, model_name, ax=axes[0, 1])

    # =====================================================
    # [0,2] Mean target theo cụm
    # =====================================================
    df_tmp = df.copy()
    df_tmp["_cluster"] = labels

    mean_target = df_tmp.groupby("_cluster")[target_col].mean()
    x_pos = np.arange(len(mean_target))

    axes[0, 2].bar(
        x_pos,
        mean_target.values,
        color=[COLORS[int(c) % len(COLORS)] for c in mean_target.index],
        edgecolor="white",
        linewidth=1.5
    )

    axes[0, 2].set_xticks(x_pos)
    axes[0, 2].set_xticklabels([f"Cluster {c}" for c in mean_target.index])

    for i, (c, val) in enumerate(zip(mean_target.index, mean_target.values)):
        axes[0, 2].text(
            i,
            val + mean_target.max() * 0.01,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

    axes[0, 2].set_title(f"Mean {target_col} theo cụm — {model_name}", fontweight="bold")
    axes[0, 2].set_ylabel(f"Mean {target_col}")
    axes[0, 2].grid(True, axis="y", alpha=0.3)

    # =====================================================
    # [1,0] Boxplot target theo cụm
    # =====================================================
    clusters = sorted(np.unique(labels))

    axes[1, 0].boxplot(
        [df_tmp[df_tmp["_cluster"] == c][target_col].values for c in clusters],
        labels=[f"Cluster {c}" for c in clusters],
        patch_artist=True,
        boxprops=dict(facecolor="#f0f0f0"),
        medianprops=dict(color="red", linewidth=2)
    )

    for idx, c in enumerate(clusters):
        vals = df_tmp[df_tmp["_cluster"] == c][target_col].values

        axes[1, 0].scatter(
            np.full(len(vals), idx + 1) + np.random.uniform(-0.15, 0.15, len(vals)),
            vals,
            alpha=0.15,
            s=6,
            color=COLORS[int(c) % len(COLORS)]
        )

    axes[1, 0].set_title(f"Phân phối {target_col} theo cụm — {model_name}", fontweight="bold")
    axes[1, 0].set_xlabel("Cluster")
    axes[1, 0].set_ylabel(target_col)
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    # =====================================================
    # [1,1] Heatmap đặc trưng theo cụm
    # =====================================================
    plot_feature_heatmap(
        df=df_for_heatmap,
        labels=labels,
        feature_names=feature_names,
        top_n=top_n_features,
        title=model_name,
        ax=axes[1, 1]
    )

    # =====================================================
    # [1,2] Bảng metric tóm tắt
    # =====================================================
    axes[1, 2].axis("off")

    # Hỗ trợ cả key của KMeans và KPrototypes
    sil = internal.get("silhouette", internal.get("silhouette_kprototypes", np.nan))
    db = internal.get("davies_bouldin", internal.get("davies_bouldin_custom", np.nan))
    ch = internal.get("calinski_harabasz", None)
    cost = internal.get("cost", internal.get("cost_", None))

    metric_text = (
        f"Internal Metrics\n"
        f"Silhouette: {sil:.4f}\n"
        f"Davies-Bouldin: {db:.4f}\n"
    )

    if ch is None:
        metric_text += "Calinski-Harabasz: Không dùng\n"
    elif isinstance(ch, float) and np.isnan(ch):
        metric_text += "Calinski-Harabasz: NaN\n"
    else:
        metric_text += f"Calinski-Harabasz: {ch:.1f}\n"

    if cost is not None:
        metric_text += f"Cost: {cost:.4f}\n"

    metric_text += (
        f"\nExternal Metrics vs {target_col}\n"
        f"ARI: {external['ari']:.4f}\n"
        f"FMI: {external['fmi']:.4f}"
    )

    axes[1, 2].text(
        0.05,
        0.75,
        metric_text,
        fontsize=13,
        va="top",
        family="monospace",
        bbox=dict(boxstyle="round", facecolor="#f7f7f7", edgecolor="#cccccc")
    )

    axes[1, 2].set_title("Tóm tắt chỉ số đánh giá", fontweight="bold")

    plt.tight_layout()
    plt.show()

    return {
        **internal,
        **{f"ext_{k}": v for k, v in external.items()}
    }

### 3. K-Means

In [78]:

class KMeansManual:
    def __init__(self, K=3, max_iter=100, random_state=42, tol=1e-4):
        self.K = K
        self.max_iter = max_iter
        self.random_state = random_state
        self.tol = tol
        self.centers = None
        self.labels_ = None
        self.n_iter_ = None
        self.cost_ = None
        self.centers_history = []
        self.labels_history = []

    # =====================================================
    # 1. Khởi tạo tâm cụm ban đầu
    # =====================================================
    def kmeans_init_centers(self, X, K=None):
        """
        Chọn ngẫu nhiên K điểm dữ liệu làm tâm cụm ban đầu.
        """
        
        if K is None:
            K = self.K
        np.random.seed(self.random_state)
        random_indices = np.random.choice(X.shape[0], K, replace=False)
        centers = X[random_indices]
        return centers

    # =====================================================
    # 2. Tính khoảng cách Euclidean
    # =====================================================
    def euclidean_distance(self, a, b):
        """
        Tính khoảng cách Euclidean giữa 1 điểm a và 1 điểm b.
        """
        return np.sqrt(np.sum((a - b) ** 2))

    # =====================================================
    # 3. Tính khoảng cách từ tất cả điểm đến các tâm cụm
    # =====================================================
    def compute_distances(self, X, centers):
        """
        Tính ma trận khoảng cách giữa tất cả điểm dữ liệu và các tâm cụm.

        Kết quả:
        D[i, k] = khoảng cách từ điểm X[i] đến tâm centers[k]
        """
        
        n_samples = X.shape[0]
        K = centers.shape[0]
        D = np.zeros((n_samples, K))
        for i in range(n_samples):
            for k in range(K):
                D[i, k] = self.euclidean_distance(X[i], centers[k])
        return D

    # =====================================================
    # 4. Tính khoảng cách giữa mọi cặp điểm
    # =====================================================

    def pairwise_euclidean_distances(self, X):
        """
        Tính ma trận khoảng cách Euclidean giữa mọi cặp điểm.
        D[i, j] = khoảng cách từ X[i] đến X[j]
        """

        X = np.asarray(X, dtype=float)
        n = X.shape[0]
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                dist = self.euclidean_distance(X[i], X[j])
                D[i, j] = dist
                D[j, i] = dist
        return D

    # =====================================================
    # 5. Gán nhãn cho từng điểm dữ liệu
    # =====================================================

    def kmeans_assign_labels(self, X, centers):
        """
        Gán mỗi điểm dữ liệu vào cụm có tâm gần nhất.
        """
        
        D = self.compute_distances(X, centers)
        labels = np.argmin(D, axis=1)
        return labels

    # =====================================================
    # 6. Cập nhật tâm cụm
    # =====================================================

    def kmeans_update_centers(self, X, labels, K=None):
        """
        Cập nhật tâm cụm mới bằng trung bình cộng
        của tất cả các điểm thuộc cụm đó.
        """

        if K is None:
            K = self.K
        centers = np.zeros((K, X.shape[1]))
        for k in range(K):
            Xk = X[labels == k]
            if len(Xk) > 0:
                centers[k] = np.mean(Xk, axis=0)
            else:
                # Nếu cụm rỗng, chọn lại ngẫu nhiên một điểm làm tâm
                centers[k] = X[np.random.choice(X.shape[0])]
        return centers

    # =====================================================
    # 7. Kiểm tra hội tụ
    # =====================================================
    def has_converged(self, centers, new_centers, tol=None):
        """
        Kiểm tra xem tâm cụm cũ và mới có gần như không đổi hay không.
        """
        
        if tol is None:
            tol = self.tol
        return np.linalg.norm(centers - new_centers) < tol

    # =====================================================
    # 8. Tính cost / inertia
    # =====================================================
    def kmeans_cost(self, X, labels, centers):
        """
        Tính tổng khoảng cách bình phương từ các điểm đến tâm cụm tương ứng.
        Đây tương đương inertia trong K-Means.
        """

        total_cost = 0
        for i in range(X.shape[0]):
            k = labels[i]
            total_cost += np.sum((X[i] - centers[k]) ** 2)
        return total_cost

    # =====================================================
    # 9. Thuật toán K-Means chính
    # =====================================================

    def kmeans(self, X, K=None, max_iter=None):
        """
        Thuật toán K-Means tự xây dựng.

        Trả về:
        - centers_history: danh sách các tâm cụm qua từng vòng lặp
        - labels_history : danh sách nhãn cụm qua từng vòng lặp
        - it             : số vòng lặp đã chạy
        """

        X = np.asarray(X, dtype=float)
        if K is None:
            K = self.K
        if max_iter is None:
            max_iter = self.max_iter

        centers = self.kmeans_init_centers(X, K)
        centers_history = [centers.copy()]
        labels_history = []

        for it in range(max_iter):
            # Bước 1: Gán nhãn
            labels = self.kmeans_assign_labels(X, centers)
            labels_history.append(labels.copy())

            # Bước 2: Cập nhật tâm cụm
            new_centers = self.kmeans_update_centers(X, labels, K)

            # Bước 3: Kiểm tra hội tụ
            if self.has_converged(centers, new_centers):
                centers = new_centers
                break
            centers = new_centers
            centers_history.append(centers.copy())

        final_cost = self.kmeans_cost(X, labels, centers)
        self.centers = centers
        self.labels_ = labels
        self.n_iter_ = it + 1
        self.cost_ = final_cost
        self.centers_history = centers_history
        self.labels_history = labels_history
        return centers_history, labels_history, it + 1

    # =====================================================
    # 10. Fit / Predict giống sklearn
    # =====================================================

    def fit(self, X):
        self.kmeans(
            X,
            K=self.K,
            max_iter=self.max_iter
        )
        return self

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

    def predict(self, X):
        """
        Gán nhãn cụm cho dữ liệu mới dựa trên centers đã học.
        """

        X = np.asarray(X, dtype=float)
        if self.centers is None:
            raise ValueError("Model chưa được fit. Hãy gọi fit(X) trước.")
        labels = self.kmeans_assign_labels(X, self.centers)
        return labels
    
    
KMeansManual.silhouette_score = lambda self, X, labels: silhouette_score_manual(X, labels)
KMeansManual.davies_bouldin_score = lambda self, X, labels: davies_bouldin_score_manual(X, labels)
KMeansManual.calinski_harabasz_score = lambda self, X, labels: calinski_harabasz_score_manual(X, labels)

### 4. K-Prototypes
- Numeric feature: cập nhật bằng mean
- Categorical feature: cập nhật bằng mode
- Distance: numeric distance + gamma * categorical mismath

In [79]:

class KPrototypesManual:
    def __init__(self, K=3, gamma=1.0, max_iter=100, random_state=42, tol=1e-4):
        self.K = K
        self.gamma = gamma
        self.max_iter = max_iter
        self.random_state = random_state
        self.tol = tol

        self.num_cols = None
        self.cat_cols = None

        self.prototypes_num = None
        self.prototypes_cat = None

        self.labels_ = None
        self.cost_ = None
        self.n_iter_ = None

        self.prototypes_num_history = []
        self.prototypes_cat_history = []
        self.labels_history = []

    # =====================================================
    # 1. Tách dữ liệu num và cat
    # =====================================================
    def split_num_cat(self, X):
        """
        Tách dữ liệu thành numeric và categorical.
        """

        X = X.copy()

        cat_cols = []
        num_cols = []

        for col in X.columns:
            unique_values = X[col].dropna().unique()

            if len(unique_values) <= 6 and np.all(np.equal(unique_values, unique_values.astype(int))):
                cat_cols.append(col)
            else:
                num_cols.append(col)

        self.num_cols = num_cols
        self.cat_cols = cat_cols

        X_num = X[num_cols].values
        X_cat = X[cat_cols].values

        return X_num, X_cat, num_cols, cat_cols

    # =====================================================
    # 2. Tính mode cho category
    # =====================================================
    def mode_value(self, arr):
        """
        Trả về giá trị xuất hiện nhiều nhất trong arr.
        """
        values, counts = np.unique(arr, return_counts=True)
        return values[np.argmax(counts)]

    # =====================================================
    # 3. Khởi tạo prototypes
    # =====================================================
    def kprototypes_init_prototypes(self, X_num, X_cat, K=None, random_state=None):
        """
        Chọn ngẫu nhiên K dòng dữ liệu thật làm prototype ban đầu.

        X_num: ma trận biến numeric
        X_cat: ma trận biến categorical
        K    : số cụm
        """
        if K is None:
            K = self.K
        if random_state is None:
            random_state = self.random_state

        np.random.seed(random_state)
        n_samples = X_num.shape[0]
        random_indices = np.random.choice(n_samples, K, replace=False)

        prototypes_num = X_num[random_indices].copy()
        prototypes_cat = X_cat[random_indices].copy()

        return prototypes_num, prototypes_cat

    # =====================================================
    # 4. Tính khoảng cách K-Prototypes
    # =====================================================
    def kprototypes_distance(self, x_num, x_cat, proto_num, proto_cat, gamma=None):
        """
        Tính khoảng cách từ 1 điểm dữ liệu đến 1 prototype.

        Numeric:
            squared Euclidean distance

        Categorical:
            đếm số thuộc tính khác nhau

        Total:
            distance = numeric_distance + gamma * categorical_distance
        """
        if gamma is None:
            gamma = self.gamma

        num_dist = np.sum((x_num - proto_num) ** 2)
        cat_dist = np.sum(x_cat != proto_cat)

        return num_dist + gamma * cat_dist

    def distance_point_to_point(self, x1_num, x1_cat, x2_num, x2_cat, gamma=None):
        """
        Khoảng cách giữa 2 điểm dữ liệu.
        Dùng cho Silhouette.
        """
        if gamma is None:
            gamma = self.gamma

        num_dist = np.sum((x1_num - x2_num) ** 2)
        cat_dist = np.sum(x1_cat != x2_cat)

        return num_dist + gamma * cat_dist

    # =====================================================
    # 5. Gán nhãn
    # =====================================================
    def kprototypes_assign_labels(self, X_num, X_cat, prototypes_num, prototypes_cat, gamma=None):
        """
        Gán mỗi điểm vào cụm có prototype gần nhất.
        """
        
        if gamma is None:
            gamma = self.gamma

        n_samples = X_num.shape[0]
        K = prototypes_num.shape[0]

        labels = np.zeros(n_samples, dtype=int)

        for i in range(n_samples):
            distances = []

            for k in range(K):
                dist = self.kprototypes_distance(
                    X_num[i],
                    X_cat[i],
                    prototypes_num[k],
                    prototypes_cat[k],
                    gamma
                )
                distances.append(dist)

            labels[i] = np.argmin(distances)

        return labels
    
    # =====================================================
    # 6. Cập nhật prototypes
    # =====================================================
    def kprototypes_update_prototypes(self, X_num, X_cat, labels, K=None):
        """
        Cập nhật prototype của từng cụm.

        Numeric:
            lấy mean

        Categorical:
            lấy mode
        """
        
        if K is None:
            K = self.K
        n_num_features = X_num.shape[1]
        n_cat_features = X_cat.shape[1]

        prototypes_num = np.zeros((K, n_num_features))
        prototypes_cat = np.empty((K, n_cat_features), dtype=object)

        for k in range(K):
            Xk_num = X_num[labels == k]
            Xk_cat = X_cat[labels == k]

            if len(Xk_num) > 0:
                if n_num_features > 0:
                    prototypes_num[k] = np.mean(Xk_num, axis=0)
                if n_cat_features > 0:
                    for j in range(n_cat_features):
                        prototypes_cat[k, j] = self.mode_value(Xk_cat[:, j])
                        
            else:
                random_id = np.random.choice(X_num.shape[0])
                if n_num_features > 0:
                    prototypes_num[k] = X_num[random_id]
                if n_cat_features > 0:
                    prototypes_cat[k] = X_cat[random_id]
        return prototypes_num, prototypes_cat

    # =====================================================
    # 7. Kiểm tra hội tụ
    # =====================================================
    def has_converged_kprototypes(self, old_num, old_cat, new_num, new_cat, tol=None):
        """
        Kiểm tra prototypes cũ và mới có thay đổi không.
        """
        if tol is None:
            tol = self.tol
        num_converged = np.linalg.norm(old_num - new_num) < tol
        cat_converged = np.array_equal(old_cat, new_cat)
        return num_converged and cat_converged

    # =====================================================
    # 8. Tính cost
    # =====================================================

    def kprototypes_cost(self, X_num, X_cat, labels, prototypes_num, prototypes_cat, gamma=None):
        """
        Tính tổng cost của K-Prototypes.
        """
        
        if gamma is None:
            gamma = self.gamma
        total_cost = 0
        for i in range(X_num.shape[0]):
            k = labels[i]

            dist = self.kprototypes_distance(
                X_num[i],
                X_cat[i],
                prototypes_num[k],
                prototypes_cat[k],
                gamma
            )
            total_cost += dist
        return total_cost

    # =====================================================
    # 9. Thuật toán K-Prototypes chính
    # =====================================================
    def kprototypes(self, X, K=None, gamma=None, max_iter=None, random_state=None):
        """
        Thuật toán K-Prototypes tự xây dựng.

        Input:
        - X: DataFrame chứa dữ liệu
        - K: số cụm
        - gamma: hệ số cân bằng phần categorical

        Output:
        - prototypes_num_history
        - prototypes_cat_history
        - labels_history
        - it + 1
        - num_cols
        - cat_cols
        - final_cost
        """

        if K is None:
            K = self.K
        if gamma is None:
            gamma = self.gamma
        if max_iter is None:
            max_iter = self.max_iter
        if random_state is None:
            random_state = self.random_state
            
        X_num, X_cat, num_cols, cat_cols = self.split_num_cat(X)
        prototypes_num, prototypes_cat = self.kprototypes_init_prototypes(
            X_num,
            X_cat,
            K,
            random_state=random_state
        )
        prototypes_num_history = [prototypes_num.copy()]
        prototypes_cat_history = [prototypes_cat.copy()]
        labels_history = []

        for it in range(max_iter):
            labels = self.kprototypes_assign_labels(
                X_num,
                X_cat,
                prototypes_num,
                prototypes_cat,
                gamma
            )
            labels_history.append(labels.copy())
            new_prototypes_num, new_prototypes_cat = self.kprototypes_update_prototypes(
                X_num,
                X_cat,
                labels,
                K
            )
            if self.has_converged_kprototypes(
                prototypes_num,
                prototypes_cat,
                new_prototypes_num,
                new_prototypes_cat
            ):
                prototypes_num = new_prototypes_num
                prototypes_cat = new_prototypes_cat
                break
            prototypes_num = new_prototypes_num
            prototypes_cat = new_prototypes_cat

            prototypes_num_history.append(prototypes_num.copy())
            prototypes_cat_history.append(prototypes_cat.copy())
        final_cost = self.kprototypes_cost(
            X_num,
            X_cat,
            labels_history[-1],
            prototypes_num,
            prototypes_cat,
            gamma
        )

        self.prototypes_num = prototypes_num
        self.prototypes_cat = prototypes_cat
        self.labels_ = labels_history[-1]
        self.cost_ = final_cost
        self.n_iter_ = it + 1

        self.prototypes_num_history = prototypes_num_history
        self.prototypes_cat_history = prototypes_cat_history
        self.labels_history = labels_history

        return (
            prototypes_num_history,
            prototypes_cat_history,
            labels_history,
            it + 1,
            num_cols,
            cat_cols,
            final_cost
        )

    def fit(self, X):
        self.kprototypes(
            X,
            K=self.K,
            gamma=self.gamma,
            max_iter=self.max_iter,
            random_state=self.random_state
        )
        return self

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

    # =====================================================
    # 10. Pairwise distance cho Silhouette
    # =====================================================
    def pairwise_distances(self, X_num, X_cat, gamma=None):
        if gamma is None:
            gamma = self.gamma

        n_samples = X_num.shape[0]
        D = np.zeros((n_samples, n_samples))

        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                dist = self.distance_point_to_point(
                    X_num[i],
                    X_cat[i],
                    X_num[j],
                    X_cat[j],
                    gamma
                )

                D[i, j] = dist
                D[j, i] = dist

        return D
    
    # =====================================================
    # 11. Silhouette cho K-Prototypes
    # =====================================================
    def silhouette_score(self, X, labels=None, gamma=None):
        """
        Silhouette riêng cho K-Prototypes.
        """

        if labels is None:
            labels = self.labels_

        if gamma is None:
            gamma = self.gamma

        labels = np.asarray(labels)

        X_num, X_cat, _, _ = self.split_num_cat(X)

        unique_clusters = np.unique(labels)
        n_samples = len(labels)

        if len(unique_clusters) < 2:
            return np.nan

        D = self.pairwise_distances(X_num, X_cat, gamma)
        silhouette_values = np.zeros(n_samples)

        for i in range(n_samples):
            current_cluster = labels[i]

            same_cluster_indices = np.where(labels == current_cluster)[0]
            same_cluster_indices = same_cluster_indices[same_cluster_indices != i]

            if len(same_cluster_indices) == 0:
                a_i = 0
            else:
                a_i = np.mean(D[i, same_cluster_indices])

            b_i = np.inf

            for cluster in unique_clusters:
                if cluster == current_cluster:
                    continue

                other_cluster_indices = np.where(labels == cluster)[0]
                mean_dist = np.mean(D[i, other_cluster_indices])

                if mean_dist < b_i:
                    b_i = mean_dist

            if max(a_i, b_i) == 0:
                silhouette_values[i] = 0
            else:
                silhouette_values[i] = (b_i - a_i) / max(a_i, b_i)

        return np.mean(silhouette_values)

    # =====================================================
    # 12. Davies-Bouldin custom cho K-Prototypes
    # =====================================================
    def davies_bouldin_score(self, X, labels=None, gamma=None):
        """
        Davies-Bouldin custom cho K-Prototypes.
        """

        if labels is None:
            labels = self.labels_

        if gamma is None:
            gamma = self.gamma

        labels = np.asarray(labels)

        X_num, X_cat, _, _ = self.split_num_cat(X)

        unique_clusters = np.unique(labels)
        K = len(unique_clusters)

        if K < 2:
            return np.nan

        prototypes_num, prototypes_cat = self.kprototypes_update_prototypes(
            X_num,
            X_cat,
            labels,
            K
        )

        S = np.zeros(K)

        for idx, cluster in enumerate(unique_clusters):
            Xk_num = X_num[labels == cluster]
            Xk_cat = X_cat[labels == cluster]

            distances = []

            for i in range(len(Xk_num)):
                dist = self.kprototypes_distance(
                    Xk_num[i],
                    Xk_cat[i],
                    prototypes_num[cluster],
                    prototypes_cat[cluster],
                    gamma
                )
                distances.append(dist)

            S[idx] = np.mean(distances)

        R = np.zeros((K, K))

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue

                cluster_i = unique_clusters[i]
                cluster_j = unique_clusters[j]

                M_ij = self.kprototypes_distance(
                    prototypes_num[cluster_i],
                    prototypes_cat[cluster_i],
                    prototypes_num[cluster_j],
                    prototypes_cat[cluster_j],
                    gamma
                )

                if M_ij == 0:
                    R[i, j] = np.inf
                else:
                    R[i, j] = (S[i] + S[j]) / M_ij

        return np.mean(np.max(R, axis=1))

    # =====================================================
    # 13. Evaluate internal
    # =====================================================
    def evaluate_internal(self, X, labels=None, gamma=None, model_name="K-Prototypes"):
        """
        Internal metrics cho K-Prototypes..
        """

        if labels is None:
            labels = self.labels_

        if gamma is None:
            gamma = self.gamma

        sil = self.silhouette_score(X, labels, gamma)
        db = self.davies_bouldin_score(X, labels, gamma)

        print(f"\n{'='*60}")
        print(f"  Internal Metrics — {model_name}")
        print(f"{'='*60}")
        print(f"  Silhouette K-Prototypes : {sil:+.4f} ")
        print(f"  Davies-Bouldin Custom   : {db:.4f} ")
        print(f"  Cost                    : {self.cost_:.4f}")
        print(f"{'='*60}\n")

        return {
            "model": model_name,
            "silhouette_kprototypes": sil,
            "davies_bouldin_custom": db,
            "cost": self.cost_
        }


### Hàm tìm k tối ưu 

Kmeans

In [80]:
def find_optimal_k_kmeans(
    X,
    k_range=range(2, 11),
    max_iter=100,
    random_state=42,
    tol=1e-4,
    plot=True
):
    """
    Tìm số cụm tối ưu cho K-Means Manual.

    - Cost/Inertia: thấp hơn tốt hơn, dùng xem Elbow
    - Silhouette: cao hơn tốt hơn
    - Davies-Bouldin: thấp hơn tốt hơn
    - Calinski-Harabasz: cao hơn tốt hơn
    """

    X = np.asarray(X, dtype=float)

    results = []

    for K in k_range:
        model = KMeansManual(
            K=K,
            max_iter=max_iter,
            random_state=random_state,
            tol=tol
        )

        labels = model.fit_predict(X)

        sil = model.silhouette_score(X, labels)
        db = model.davies_bouldin_score(X, labels)
        ch = model.calinski_harabasz_score(X, labels)
        cost = model.cost_

        results.append({
            "K": K,
            "cost": cost,
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch,
            "n_iter": model.n_iter_
        })

        print(
            f"K={K} | Cost={cost:.4f} | "
            f"Silhouette={sil:.4f} | DBI={db:.4f} | CH={ch:.2f}"
        )

    results_df = pd.DataFrame(results)

    if plot:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Optimal K — K-Means", fontsize=15, fontweight="bold")

        axes[0, 0].plot(results_df["K"], results_df["cost"], marker="o")
        axes[0, 0].set_title("Elbow Method — Cost/Inertia")
        axes[0, 0].set_xlabel("K")
        axes[0, 0].set_ylabel("Cost")
        axes[0, 0].grid(alpha=0.3)

        axes[0, 1].plot(results_df["K"], results_df["silhouette"], marker="o")
        axes[0, 1].set_title("Silhouette Score")
        axes[0, 1].set_xlabel("K")
        axes[0, 1].set_ylabel("Silhouette")
        axes[0, 1].grid(alpha=0.3)

        axes[1, 0].plot(results_df["K"], results_df["davies_bouldin"], marker="o")
        axes[1, 0].set_title("Davies-Bouldin Index")
        axes[1, 0].set_xlabel("K")
        axes[1, 0].set_ylabel("DBI")
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].plot(results_df["K"], results_df["calinski_harabasz"], marker="o")
        axes[1, 1].set_title("Calinski-Harabasz Score")
        axes[1, 1].set_xlabel("K")
        axes[1, 1].set_ylabel("CH")
        axes[1, 1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    return results_df

K-propeypes

In [81]:
def find_optimal_k_gamma_kprototypes(
    X,
    k_range=range(2, 11),
    gamma_range=None,
    max_iter=100,
    random_state=42,
    tol=1e-4,
    plot=True
):
    """
    Tìm K và gamma tối ưu cho K-Prototypes Manual.

    Test đồng thời:
    - K: số cụm
    - gamma: hệ số cân bằng numeric và categorical

    Metrics:
    - Cost: càng thấp càng tốt
    - Silhouette K-Prototypes: càng cao càng tốt
    - Davies-Bouldin Custom: càng thấp càng tốt
    """

    if gamma_range is None:
        gamma_range = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0]

    results = []

    for gamma in gamma_range:
        print("\n" + "=" * 80)
        print(f"Testing gamma = {gamma}")
        print("=" * 80)

        for K in k_range:
            model = KPrototypesManual(
                K=K,
                gamma=gamma,
                max_iter=max_iter,
                random_state=random_state,
                tol=tol
            )

            labels = model.fit_predict(X)

            cost = model.cost_

            sil = model.silhouette_score(
                X=X,
                labels=labels,
                gamma=gamma
            )

            db = model.davies_bouldin_score(
                X=X,
                labels=labels,
                gamma=gamma
            )

            cluster_sizes = pd.Series(labels).value_counts().sort_index()

            results.append({
                "K": K,
                "gamma": gamma,
                "cost": cost,
                "silhouette_kprototypes": sil,
                "davies_bouldin_custom": db,
                "n_iter": model.n_iter_,
                "min_cluster_size": cluster_sizes.min(),
                "max_cluster_size": cluster_sizes.max(),
                "cluster_size_ratio": cluster_sizes.max() / cluster_sizes.min(),
                "num_cols": model.num_cols,
                "cat_cols": model.cat_cols
            })

            print(
                f"K={K:2d} | gamma={gamma:<4} | "
                f"Cost={cost:.4f} | "
                f"Sil={sil:.4f} | "
                f"DBI={db:.4f} | "
                f"size_ratio={cluster_sizes.max() / cluster_sizes.min():.2f}"
            )

    results_df = pd.DataFrame(results)

    # Sắp xếp gợi ý:
    # Silhouette cao, DBI thấp, cụm không quá lệch
    results_df["rank_silhouette"] = results_df["silhouette_kprototypes"].rank(
        ascending=False
    )

    results_df["rank_dbi"] = results_df["davies_bouldin_custom"].rank(
        ascending=True
    )

    results_df["rank_balance"] = results_df["cluster_size_ratio"].rank(
        ascending=True
    )

    results_df["final_rank"] = (
        results_df["rank_silhouette"]
        + results_df["rank_dbi"]
        + 0.5 * results_df["rank_balance"]
    )

    results_df = results_df.sort_values("final_rank").reset_index(drop=True)

    best_row = results_df.iloc[0]

    print("\n" + "=" * 80)
    print("BEST RESULT")
    print("=" * 80)
    print(f"Best K      : {best_row['K']}")
    print(f"Best gamma  : {best_row['gamma']}")
    print(f"Cost        : {best_row['cost']:.4f}")
    print(f"Silhouette  : {best_row['silhouette_kprototypes']:.4f}")
    print(f"DBI         : {best_row['davies_bouldin_custom']:.4f}")
    print(f"Size ratio  : {best_row['cluster_size_ratio']:.4f}")

    if plot:
        # Lấy mean theo K để nhìn xu hướng tổng quát
        by_k = results_df.groupby("K").agg({
            "cost": "mean",
            "silhouette_kprototypes": "mean",
            "davies_bouldin_custom": "mean"
        }).reset_index()

        # Lấy mean theo gamma
        by_gamma = results_df.groupby("gamma").agg({
            "cost": "mean",
            "silhouette_kprototypes": "mean",
            "davies_bouldin_custom": "mean"
        }).reset_index()

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle("Optimal K and Gamma — K-Prototypes", fontsize=15, fontweight="bold")

        # Theo K
        axes[0, 0].plot(by_k["K"], by_k["cost"], marker="o")
        axes[0, 0].set_title("Average Cost by K")
        axes[0, 0].set_xlabel("K")
        axes[0, 0].set_ylabel("Cost")
        axes[0, 0].grid(alpha=0.3)

        axes[0, 1].plot(by_k["K"], by_k["silhouette_kprototypes"], marker="o")
        axes[0, 1].set_title("Average Silhouette by K")
        axes[0, 1].set_xlabel("K")
        axes[0, 1].set_ylabel("Silhouette")
        axes[0, 1].grid(alpha=0.3)

        axes[0, 2].plot(by_k["K"], by_k["davies_bouldin_custom"], marker="o")
        axes[0, 2].set_title("Average DBI by K")
        axes[0, 2].set_xlabel("K")
        axes[0, 2].set_ylabel("DBI")
        axes[0, 2].grid(alpha=0.3)

        # Theo gamma
        axes[1, 0].plot(by_gamma["gamma"], by_gamma["cost"], marker="o")
        axes[1, 0].set_title("Average Cost by Gamma")
        axes[1, 0].set_xlabel("Gamma")
        axes[1, 0].set_ylabel("Cost")
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].plot(by_gamma["gamma"], by_gamma["silhouette_kprototypes"], marker="o")
        axes[1, 1].set_title("Average Silhouette by Gamma")
        axes[1, 1].set_xlabel("Gamma")
        axes[1, 1].set_ylabel("Silhouette")
        axes[1, 1].grid(alpha=0.3)

        axes[1, 2].plot(by_gamma["gamma"], by_gamma["davies_bouldin_custom"], marker="o")
        axes[1, 2].set_title("Average DBI by Gamma")
        axes[1, 2].set_xlabel("Gamma")
        axes[1, 2].set_ylabel("DBI")
        axes[1, 2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    return results_df

## 6. Giảm chiều

In [82]:
# PCA: chỉ áp dụng cho biến liên tục, ghép lại biến rời rạc
pca = (n_components=0.70, random_state=42)
X_cont_pca = pca.fit_transform(X_cont)
X_pca_reduced = np.hstack([X_cont_pca, X_disc.values])
pca_col_names = ([f'PC{i+1}' for i in range(X_cont_pca.shape[1])] + list(X_disc.columns))
X_pca_reduced = pd.DataFrame(X_pca_reduced, columns=pca_col_names, index=X.index)
print(f"PCA: {X_cont.shape[1]}D → {X_cont_pca.shape[1]}D + {X_disc.shape[1]}D rời rạc "
      f"= {X_pca_reduced.shape[1]}D | variance: {pca.explained_variance_ratio_.sum():.2%}")

# UMAP: áp dụng toàn bộ
reducer = umap.UMAP(n_components=15, random_state=42)
X_umap_arr = reducer.fit_transform(X)
umap_col_names = [f'UMAP{i+1}' for i in range(X_umap_arr.shape[1])]
X_umap_reduced = pd.DataFrame(X_umap_arr, columns=umap_col_names, index=X.index)
print(f"UMAP: {X.shape[1]}D → {X_umap_reduced.shape[1]}D")

PCA: 25D → 13D + 21D rời rạc = 34D | variance: 71.79%
UMAP: 46D → 15D


## TEST

In [83]:
# ============================================================
# FULL TEST PIPELINE CLUSTERING
# Dùng đúng các hàm / class đã viết trong notebook
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 0. FIX NHẸ CHO KMeansManual
# Vì find_optimal_k_kmeans đang gọi:
# model.silhouette_score, model.davies_bouldin_score,
# model.calinski_harabasz_score
# Nhưng trong class KMeansManual hiện chưa có 3 method này.
# Ta gắn lại bằng các hàm manual mày đã viết ở ngoài.
# ============================================================

KMeansManual.silhouette_score = lambda self, X, labels: silhouette_score_manual(X, labels)
KMeansManual.davies_bouldin_score = lambda self, X, labels: davies_bouldin_score_manual(X, labels)
KMeansManual.calinski_harabasz_score = lambda self, X, labels: calinski_harabasz_score_manual(X, labels)


# ============================================================
# 1. HÀM CHỌN BEST K CHO KMEANS
# ============================================================

def choose_best_k_kmeans(optimal_df):
    """
    Chọn K tốt cho KMeans dựa trên:
    - silhouette cao
    - DBI thấp
    - CH cao
    """

    df_rank = optimal_df.copy()

    df_rank["rank_sil"] = df_rank["silhouette"].rank(ascending=False)
    df_rank["rank_dbi"] = df_rank["davies_bouldin"].rank(ascending=True)
    df_rank["rank_ch"] = df_rank["calinski_harabasz"].rank(ascending=False)

    df_rank["final_rank"] = (
        df_rank["rank_sil"]
        + df_rank["rank_dbi"]
        + df_rank["rank_ch"]
    )

    df_rank = df_rank.sort_values("final_rank").reset_index(drop=True)

    best_K = int(df_rank.loc[0, "K"])

    print("\nBest KMeans K:", best_K)
    display(df_rank.head(10))

    return best_K, df_rank


# ============================================================
# 2. HÀM CHỌN BEST K + GAMMA CHO K-PROTOTYPES
# ============================================================

def choose_best_k_gamma_kprototypes(optimal_df):
    """
    find_optimal_k_gamma_kprototypes đã có final_rank rồi.
    Hàm này chỉ lấy dòng tốt nhất.
    """

    df_rank = optimal_df.copy()

    if "final_rank" not in df_rank.columns:
        df_rank["rank_sil"] = df_rank["silhouette_kprototypes"].rank(ascending=False)
        df_rank["rank_dbi"] = df_rank["davies_bouldin_custom"].rank(ascending=True)
        df_rank["rank_balance"] = df_rank["cluster_size_ratio"].rank(ascending=True)

        df_rank["final_rank"] = (
            df_rank["rank_sil"]
            + df_rank["rank_dbi"]
            + 0.5 * df_rank["rank_balance"]
        )

    df_rank = df_rank.sort_values("final_rank").reset_index(drop=True)

    best_K = int(df_rank.loc[0, "K"])
    best_gamma = float(df_rank.loc[0, "gamma"])

    print("\nBest K-Prototypes K:", best_K)
    print("Best K-Prototypes gamma:", best_gamma)
    display(df_rank.head(10))

    return best_K, best_gamma, df_rank


# ============================================================
# 3. HÀM FIT + REPORT KMEANS
# ============================================================

def test_kmeans_case(
    X_cluster,
    df,
    df_for_heatmap,
    model_name,
    k_range=range(2, 8),
    target_col="charges",
    use_umap=False,
    max_iter=100,
    random_state=42,
    tol=1e-4,
    top_n_features=20
):
    """
    Test 1 case KMeans:
    - Tìm K tối ưu
    - Fit KMeansManual
    - In metrics
    - Vẽ full_cluster_report
    """

    print("\n" + "#" * 90)
    print(model_name)
    print("#" * 90)

    # 1. Tìm K
    optimal_df = find_optimal_k_kmeans(
        X=X_cluster,
        k_range=k_range,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        plot=True
    )

    best_K, ranked_df = choose_best_k_kmeans(optimal_df)

    # 2. Fit model cuối
    model = KMeansManual(
        K=best_K,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol
    )

    labels = model.fit_predict(X_cluster)

    # 3. Internal metrics
    internal = evaluate_internal(
        X_cluster,
        labels,
        model_name=model_name
    )

    # 4. Full report
    report = full_cluster_report(
        X=X_cluster,
        df=df,
        labels=labels,
        km_model=model,
        feature_names=df_for_heatmap.columns.tolist(),
        df_for_heatmap=df_for_heatmap,
        target_col=target_col,
        model_name=f"{model_name} | K={best_K}",
        use_umap=use_umap,
        top_n_features=top_n_features,
        internal=internal,
        n_bins=3
    )

    return {
        "model_name": model_name,
        "model": model,
        "labels": labels,
        "optimal_df": optimal_df,
        "ranked_df": ranked_df,
        "best_K": best_K,
        "internal": internal,
        "report": report
    }


# ============================================================
# 4. HÀM FIT + REPORT K-PROTOTYPES
# ============================================================

def test_kprototypes_case(
    X_cluster_df,
    df,
    df_for_heatmap,
    model_name,
    k_range=range(2, 8),
    gamma_range=None,
    target_col="charges",
    max_iter=100,
    random_state=42,
    tol=1e-4,
    top_n_features=20
):
    """
    Test 1 case K-Prototypes:
    - Tìm K + gamma tối ưu
    - Fit KPrototypesManual
    - In metrics
    - Vẽ full_cluster_report
    """

    print("\n" + "#" * 90)
    print(model_name)
    print("#" * 90)

    if gamma_range is None:
        gamma_range = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]

    # 1. Tìm K + gamma
    optimal_df = find_optimal_k_gamma_kprototypes(
        X=X_cluster_df,
        k_range=k_range,
        gamma_range=gamma_range,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        plot=True
    )

    best_K, best_gamma, ranked_df = choose_best_k_gamma_kprototypes(optimal_df)

    # 2. Fit model cuối
    model = KPrototypesManual(
        K=best_K,
        gamma=best_gamma,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol
    )

    labels = model.fit_predict(X_cluster_df)

    # 3. Internal metrics riêng của K-Prototypes
    internal = model.evaluate_internal(
        X=X_cluster_df,
        labels=labels,
        gamma=best_gamma,
        model_name=model_name
    )

    # 4. Full report
    # Với K-Prototypes, km_model=None vì không có centroid kiểu KMeans để vẽ.
    report = full_cluster_report(
        X=X_cluster_df.values.astype(float),
        df=df,
        labels=labels,
        km_model=None,
        feature_names=df_for_heatmap.columns.tolist(),
        df_for_heatmap=df_for_heatmap,
        target_col=target_col,
        model_name=f"{model_name} | K={best_K}, gamma={best_gamma}",
        use_umap=False,
        top_n_features=top_n_features,
        internal=internal,
        n_bins=3
    )

    return {
        "model_name": model_name,
        "model": model,
        "labels": labels,
        "optimal_df": optimal_df,
        "ranked_df": ranked_df,
        "best_K": best_K,
        "best_gamma": best_gamma,
        "internal": internal,
        "report": report
    }


# ============================================================
# 5. FULL PIPELINE ĐÚNG YÊU CẦU
# ============================================================

def run_full_clustering_test_pipeline(
    df,
    target_col="charges",
    k_range=range(2, 8),
    gamma_range=None,
    pca_ratio=1/3,
    umap_ratio=1/3,
    max_iter=100,
    random_state=42,
    tol=1e-4,
    top_n_features=20
):
    """
    Full pipeline:

    1. Raw data:
       - KMeansManual
       - KPrototypesManual

    2. PCA:
       - KMeansManual trên PCAmix toàn bộ dữ liệu
       - KPrototypesManual trên PCA phần continuous + discrete gốc

    3. UMAP:
       - KMeansManual trên UMAP
       - Không chạy KPrototypes trên UMAP

    Tất cả đều:
       - tìm K / gamma
       - fit model
       - đánh giá internal
       - đánh giá external với charges
       - vẽ full_cluster_report
    """

    results = {}

    # ========================================================
    # 5.1. Chuẩn bị dữ liệu gốc
    # ========================================================

    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()

    feature_names = X.columns.tolist()

    continuous_cols, discrete_cols = infer_discrete_continuous_cols(X)

    X_cont = X[continuous_cols].copy()
    X_disc = X[discrete_cols].copy()

    print("\n" + "=" * 90)
    print("DATA INFO")
    print("=" * 90)
    print("Shape df:", df.shape)
    print("Shape X :", X.shape)
    print("Continuous columns:", len(continuous_cols))
    print("Discrete columns  :", len(discrete_cols))
    print("Target:", target_col)

    # index cho PCAmix
    continuous_idx = [X.columns.get_loc(c) for c in continuous_cols]
    discrete_idx = [X.columns.get_loc(c) for c in discrete_cols]

    X_np = X.values.astype(float)

    # ========================================================
    # 1. RAW DATA
    # ========================================================

    print("\n\n" + "=" * 90)
    print("1. RAW DATA")
    print("=" * 90)

    # --------------------------------------------------------
    # 1.1 KMeans trên dữ liệu gốc 47D
    # --------------------------------------------------------

    results["raw_kmeans"] = test_kmeans_case(
        X_cluster=X_np,
        df=df,
        df_for_heatmap=X,
        model_name="RAW DATA - KMeansManual",
        k_range=k_range,
        target_col=target_col,
        use_umap=False,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        top_n_features=top_n_features
    )

    # --------------------------------------------------------
    # 1.2 K-Prototypes trên dữ liệu gốc
    # --------------------------------------------------------

    results["raw_kprototypes"] = test_kprototypes_case(
        X_cluster_df=X,
        df=df,
        df_for_heatmap=X,
        model_name="RAW DATA - KPrototypesManual",
        k_range=k_range,
        gamma_range=gamma_range,
        target_col=target_col,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        top_n_features=top_n_features
    )

    # ========================================================
    # 2. PCA DATA
    # ========================================================

    print("\n\n" + "=" * 90)
    print("2. PCA DATA")
    print("=" * 90)

    # --------------------------------------------------------
    # 2.1 KMeans trên PCAmix toàn bộ dữ liệu
    # --------------------------------------------------------

    n_components_mixed = max(2, int(X.shape[1] * pca_ratio))

    pcamix = PCAmix_Manual(
        n_components=n_components_mixed,
        discrete_features=discrete_idx,
        continuous_features=continuous_idx
    )

    X_pcamix = pcamix.fit_transform(X_np)

    print("\nPCAmix shape:", X_pcamix.shape)
    print("PCAmix explained variance sum:", np.sum(pcamix.explained_variance_ratio_))

    results["pca_mixed_kmeans"] = test_kmeans_case(
        X_cluster=X_pcamix,
        df=df,
        df_for_heatmap=X,
        model_name="PCA MIXED - KMeansManual",
        k_range=k_range,
        target_col=target_col,
        use_umap=False,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        top_n_features=top_n_features
    )

    # --------------------------------------------------------
    # 2.2 K-Prototypes:
    # PCA cho continuous, sau đó ghép discrete gốc
    # --------------------------------------------------------

    n_components_cont = max(2, int(len(continuous_cols) * pca_ratio))

    pca_cont = PCA_Manual(
        n_components=n_components_cont
    )

    X_cont_pca = pca_cont.fit_transform(X_cont.values.astype(float))

    pca_cont_cols = [
        f"PC_cont_{i+1}"
        for i in range(X_cont_pca.shape[1])
    ]

    X_cont_pca_df = pd.DataFrame(
        X_cont_pca,
        columns=pca_cont_cols,
        index=X.index
    )

    X_pca_kproto = pd.concat(
        [
            X_cont_pca_df,
            X_disc.reset_index(drop=True)
        ],
        axis=1
    )

    print("\nPCA continuous shape:", X_cont_pca_df.shape)
    print("PCA continuous explained variance sum:", np.sum(pca_cont.explained_variance_ratio_))
    print("PCA continuous + discrete shape:", X_pca_kproto.shape)

    results["pca_cont_kprototypes"] = test_kprototypes_case(
        X_cluster_df=X_pca_kproto,
        df=df,
        df_for_heatmap=X,
        model_name="PCA CONTINUOUS + DISCRETE - KPrototypesManual",
        k_range=k_range,
        gamma_range=gamma_range,
        target_col=target_col,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        top_n_features=top_n_features
    )

    # ========================================================
    # 3. UMAP DATA
    # ========================================================

    print("\n\n" + "=" * 90)
    print("3. UMAP DATA")
    print("=" * 90)

    n_components_umap = max(2, int(X.shape[1] * umap_ratio))

    reducer = umap.UMAP(
        n_components=n_components_umap,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=random_state
    )

    X_umap = reducer.fit_transform(X_np)

    print("\nUMAP shape:", X_umap.shape)

    # --------------------------------------------------------
    # 3.1 KMeans trên UMAP
    # --------------------------------------------------------

    results["umap_kmeans"] = test_kmeans_case(
        X_cluster=X_umap,
        df=df,
        df_for_heatmap=X,
        model_name="UMAP - KMeansManual",
        k_range=k_range,
        target_col=target_col,
        use_umap=True,
        max_iter=max_iter,
        random_state=random_state,
        tol=tol,
        top_n_features=top_n_features
    )

    print("\nBỏ qua K-Prototypes trên UMAP vì UMAP đã biến toàn bộ dữ liệu sang không gian số thực liên tục.")

    # ========================================================
    # 4. Tổng hợp kết quả
    # ========================================================

    summary_rows = []

    for key, value in results.items():
        row = {
            "case": key,
            "model_name": value["model_name"],
            "best_K": value.get("best_K", None),
            "best_gamma": value.get("best_gamma", None)
        }

        internal = value.get("internal", {})

        for metric_name, metric_value in internal.items():
            row[metric_name] = metric_value

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    print("\n\n" + "=" * 90)
    print("FINAL SUMMARY")
    print("=" * 90)
    display(summary_df)

    return results, summary_df

In [ ]:
results, summary_df = run_full_clustering_test_pipeline(
    df=df,
    target_col="charges",
    k_range=range(2, 5),
    gamma_range=[0.3, 0.7, 1.0],
    pca_ratio=1/3,
    umap_ratio=1/3,
    max_iter=50,
    random_state=42,
    tol=1e-4,
    top_n_features=15
)


DATA INFO
Shape df: (8933, 47)
Shape X : (8933, 46)
Continuous columns: 25
Discrete columns  : 21
Target: charges


1. RAW DATA

##########################################################################################
RAW DATA - KMeansManual
##########################################################################################
